# Coimbra -- Unified, Publication-Quality Figure Generation

Consolidates and replaces the plotting code across `coimbra_road_analysis.ipynb`,
`coimbra_run.ipynb`, and `Copy_of_coimbra_road_analysis.ipynb` with a single,
consistent toolkit addressing the following reviewer comments directly:

- **REV 01-06 / REV 05-04**: maps too small/abstract, no real road/block context,
  inconsistent labels/legends/colour scales/line widths/boundaries/scale bars/titles.
- **REV 03-05**: all figures/tables need font sizes and layout suitable for A4 print.
- **REV 04-01 (done)**: overview figure relocated to its own subsection, captioned
  with a source line -- this notebook produces that artifact.
- **REV 04-011**: figure text (esp. the critical-nodes figure, "Figure 13") must
  be legible, not just present.
- **REV 04-012**: named areas mentioned throughout the text need to appear in a figure.
- **REV 04-013**: the centrality-overview figure needs a legend / expanded caption.

**Fixes applied uniformly to every figure below:**
1. One shared plotting function (`plot_structural_map`) used for every geographic
   figure, instead of ad hoc, inconsistent code per figure.
2. A4-print-safe font sizes (title 16pt bold, axis/legend 12-13pt, colorbar label 13pt).
3. City administrative boundary overlaid on every geographic map.
4. Scale bar and north arrow on every geographic map.
5. Properly labelled colour scale (colorbar with units/meaning) or a real legend
   with named categories -- never colour alone with no explanation.
6. Larger, print-safe node sizes and edge line widths (no more `node_size=6`,
   `edge_linewidth=0.3`).
7. Optional real basemap (light street/block tiles via `contextily`) underneath
   the network, so results read against actual urban fabric, not abstract lines.
8. All figures saved at 300 DPI (several `savefig` calls in the original
   notebooks were commented out entirely).
9. A dedicated "Named Areas" overview figure, annotating the same districts
   referenced throughout the text (Baixa, Alta/University, Solum, Celas, etc.).
10. Every figure's suggested LaTeX caption includes an explicit source line.

In [ ]:
# !pip install osmnx networkx matplotlib-scalebar contextily --quiet

import osmnx as ox
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib_scalebar.scalebar import ScaleBar
import os
import random
import time

ox.settings.use_cache = True
ox.settings.log_console = False

PLACE = "Coimbra, Portugal"
CMAP = "viridis"
OUTPUT_DIR = "figures_coimbra"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------
# Global, A4-print-safe font sizes (REV 03-05). Applied via rcParams so
# every figure below inherits these consistently, rather than each plot
# function setting its own (previously inconsistent) sizes.
# ---------------------------------------------------------------
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "figure.dpi": 150,   # on-screen preview; saved files use 300 explicitly
})
SAVE_DPI = 300

In [ ]:
# ---------------------------------------------------------------
# Extraction (reused pattern from the main analysis pipeline)
# ---------------------------------------------------------------
t0 = time.time()
G_raw = ox.graph_from_place(PLACE, network_type="drive")
G_multi = ox.convert.to_undirected(G_raw)
largest_cc = max(nx.connected_components(G_multi), key=len)
G_multi = G_multi.subgraph(largest_cc).copy()
G = nx.Graph(G_multi)  # simple graph for algorithms that need it

print(f"Extracted in {time.time()-t0:.1f}s -- {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# ---------------------------------------------------------------
# Fetch the administrative city boundary once -- overlaid on every map below
# ---------------------------------------------------------------
try:
    city_boundary = ox.geocode_to_gdf(PLACE)
except Exception as e:
    print(f"Boundary fetch failed: {e}")
    city_boundary = None

In [ ]:
# ---------------------------------------------------------------
# Compute the metrics needed for the figures below.
# Betweenness uses approximate (pivot-sampled) computation for speed,
# consistent with the approach used elsewhere in this study.
# ---------------------------------------------------------------
t0 = time.time()
degree_dict = dict(G.degree())
closeness_dict = nx.closeness_centrality(G)
betweenness_dict = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), seed=42, normalized=True)
eigenvector_dict = nx.eigenvector_centrality(G, max_iter=1000)
pagerank_dict = nx.pagerank(G, alpha=0.85)
print(f"Node metrics computed in {time.time()-t0:.1f}s")

t0 = time.time()
louvain_communities = nx.community.louvain_communities(G, seed=42)
community_dict = {}
for cid, nodes_in_comm in enumerate(louvain_communities):
    for n in nodes_in_comm:
        community_dict[n] = cid
print(f"Louvain ({len(louvain_communities)} communities) computed in {time.time()-t0:.1f}s")

t0 = time.time()
bridges = list(nx.bridges(G))
edge_betweenness = nx.edge_betweenness_centrality(G)
weak_ties = set(sorted(edge_betweenness, key=edge_betweenness.get)[:50])
print(f"Bridges ({len(bridges)}) and weak ties (50) computed in {time.time()-t0:.1f}s")

## Shared plotting toolkit

Every geographic figure below calls this one function, so labels, legends,
colour scales, line widths, boundaries, scale bars, and titles are consistent
across all figures -- directly addressing REV 01-06 and REV 03-05.

In [ ]:
def plot_structural_map(G_multi, node_values=None, categorical_groups=None,
                         title="", value_label="", cmap_name=CMAP,
                         boundary=None, basemap=False, figsize=(11, 11),
                         node_size_range=(4, 60), edge_linewidth=0.7,
                         save_path=None, legend_loc="lower left"):
    """
    Unified geographic network plot.

    node_values        : dict {node: float} for a continuous colour-scaled map
                          (e.g. betweenness, closeness). Mutually exclusive with
                          categorical_groups.
    categorical_groups  : dict {group_label: (set_of_nodes, color)} for a
                          categorical map with a real legend (e.g. critical
                          nodes by type, bridges vs weak ties). Mutually
                          exclusive with node_values.
    boundary            : a GeoDataFrame (e.g. city_boundary) to overlay.
    basemap             : if True, adds a light real-street/block basemap
                          via contextily underneath the network.
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Base network (always drawn first, as light context)
    ox.plot_graph(G_multi, ax=ax, node_size=0, edge_color="#B0B0B0",
                  edge_linewidth=edge_linewidth * 0.6, show=False, close=False,
                  bgcolor="white")

    if node_values is not None:
        vals = np.array(list(node_values.values()))
        norm = mcolors.Normalize(vmin=vals.min(), vmax=vals.max())
        cmap = cm.get_cmap(cmap_name)
        nodes = list(G_multi.nodes())
        colors = [cmap(norm(node_values.get(n, 0))) for n in nodes]
        lo, hi = node_size_range
        sizes = lo + (vals - vals.min()) / (vals.ptp() + 1e-9) * (hi - lo)
        xs = [G_multi.nodes[n]['x'] for n in nodes]
        ys = [G_multi.nodes[n]['y'] for n in nodes]
        ax.scatter(xs, ys, c=colors, s=sizes, zorder=3, edgecolors='none')

        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.65, pad=0.02)
        cbar.set_label(value_label if value_label else "Value", fontsize=13)
        cbar.ax.tick_params(labelsize=11)

    if categorical_groups is not None:
        for label, (node_set, color) in categorical_groups.items():
            xs = [G_multi.nodes[n]['x'] for n in node_set if n in G_multi.nodes]
            ys = [G_multi.nodes[n]['y'] for n in node_set if n in G_multi.nodes]
            if xs:
                ax.scatter(xs, ys, c=color, s=40, label=label, zorder=4,
                          edgecolors='black', linewidths=0.4)
        ax.legend(loc=legend_loc, fontsize=12, framealpha=0.9, title="Legend",
                  title_fontsize=13)

    if boundary is not None:
        boundary.boundary.plot(ax=ax, color='black', linewidth=1.4,
                               linestyle='--', zorder=5)

    if basemap:
        try:
            import contextily as cx
            cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron, alpha=0.55, zorder=0)
        except Exception as e:
            print(f"Basemap failed (continuing without it): {e}")

    # Scale bar (metres, since coordinates are lat/lon -- approximate but adequate at city scale)
    try:
        ax.add_artist(ScaleBar(85000, units="m", dimension="si-length",
                               location='lower right', box_alpha=0.75,
                               font_properties={'size': 11}))
    except Exception as e:
        print(f"Scale bar failed: {e}")

    # North arrow
    ax.annotate('N', xy=(0.95, 0.12), xytext=(0.95, 0.04), xycoords='axes fraction',
               fontsize=15, fontweight='bold', ha='center',
               arrowprops=dict(facecolor='black', width=3, headwidth=9))

    ax.set_title(title, fontsize=16, fontweight='bold', pad=12)
    ax.set_axis_off()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()
    return fig, ax

## 1. Overview: Named Areas (relocated "Figure 1" + REV 04-012 area labels)

Addresses REV 04-01 (Figure 1 relocated to its own subsection with a legend
and citeable source) and REV 04-012 (named areas referenced throughout the
text should appear in a figure).

In [ ]:
# Named areas referenced throughout the manuscript's discussion of Coimbra
# (coordinates from the critical-node analysis reported elsewhere in this study)
NAMED_AREAS = {
    "Baixa":                    (40.2056, -8.4196),
    "University / Alta":        (40.2080, -8.4265),
    "Av. Fern\u00e3o de Magalh\u00e3es": (40.2072, -8.4160),
    "Pra\u00e7a da Rep\u00fablica":        (40.2039, -8.4212),
    "Santa Clara":              (40.2095, -8.4238),
    "Solum":                    (40.2063, -8.4180),
    "Montes Claros":            (40.2048, -8.4171),
    "Celas":                    (40.2079, -8.4204),
    "Rua da Sofia":             (40.2068, -8.4226),
    "Portagem":                 (40.2052, -8.4189),
}

fig, ax = plt.subplots(figsize=(12, 12))
ox.plot_graph(G_multi, ax=ax, node_size=0, edge_color="#999999",
              edge_linewidth=0.5, show=False, close=False, bgcolor="white")

if city_boundary is not None:
    city_boundary.boundary.plot(ax=ax, color='black', linewidth=1.4, linestyle='--', zorder=5)

for name, (lat, lon) in NAMED_AREAS.items():
    ax.scatter(lon, lat, c='crimson', s=90, zorder=6, edgecolors='black', linewidths=0.6)
    ax.annotate(name, xy=(lon, lat), xytext=(6, 6), textcoords='offset points',
                fontsize=12, fontweight='bold', color='black',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85, edgecolor='none'),
                zorder=7)

try:
    ax.add_artist(ScaleBar(85000, units="m", dimension="si-length",
                           location='lower right', box_alpha=0.75, font_properties={'size': 11}))
except Exception as e:
    print(f"Scale bar failed: {e}")

ax.annotate('N', xy=(0.95, 0.12), xytext=(0.95, 0.04), xycoords='axes fraction',
            fontsize=15, fontweight='bold', ha='center',
            arrowprops=dict(facecolor='black', width=3, headwidth=9))

ax.set_title("Coimbra -- Named Areas Referenced Throughout This Study", fontsize=16, fontweight='bold', pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/coimbra_named_areas.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

print("""
Suggested LaTeX caption:
"Named areas of Coimbra referenced throughout Sections 4-5, shown on the
extracted road network with the municipal boundary overlaid. Source:
authors' own elaboration, generated using Python (NetworkX/OSMnx) from
OpenStreetMap data (\\copyright{} OpenStreetMap contributors)."
""")

## 2. Betweenness Centrality Map (with basemap; addresses REV 04-013 legend request)

In [ ]:
plot_structural_map(
    G_multi, node_values=betweenness_dict,
    title="Coimbra -- Betweenness Centrality",
    value_label="Betweenness centrality (0 = low, 1 = high)",
    boundary=city_boundary, basemap=True,
    save_path=f"{OUTPUT_DIR}/coimbra_betweenness.png"
)
print("""
Suggested LaTeX caption:
"Betweenness centrality of Coimbra's road network, overlaid on the municipal
boundary and a real street/block basemap. Colour indicates each intersection's
betweenness centrality, from low (dark) to high (yellow), following a viridis
colour scale with an explicit legend. Source: authors' own elaboration,
generated using Python (NetworkX/OSMnx) from OpenStreetMap data
(\\copyright{} OpenStreetMap contributors)."
""")

## 3. Closeness, Eigenvector, and PageRank Maps

In [ ]:
plot_structural_map(
    G_multi, node_values=closeness_dict,
    title="Coimbra -- Closeness Centrality",
    value_label="Closeness centrality",
    boundary=city_boundary, basemap=True,
    save_path=f"{OUTPUT_DIR}/coimbra_closeness.png"
)

In [ ]:
plot_structural_map(
    G_multi, node_values=eigenvector_dict,
    title="Coimbra -- Eigenvector Centrality",
    value_label="Eigenvector centrality",
    boundary=city_boundary, basemap=True,
    save_path=f"{OUTPUT_DIR}/coimbra_eigenvector.png"
)

In [ ]:
plot_structural_map(
    G_multi, node_values=pagerank_dict,
    title="Coimbra -- PageRank",
    value_label="PageRank score",
    boundary=city_boundary, basemap=True,
    save_path=f"{OUTPUT_DIR}/coimbra_pagerank.png"
)

## 4. Community Structure (previously had NO legend at all -- fixed here)

In [ ]:
# Build a categorical group per community, with a distinguishable colour each.
# Only the largest communities get individual legend entries (a 60+ entry
# legend would be unreadable); smaller ones are grouped as "Other communities".
sizes = [(cid, len(nodes)) for cid, nodes in enumerate(louvain_communities)]
sizes.sort(key=lambda x: -x[1])
top_community_ids = [cid for cid, _ in sizes[:8]]  # largest 8 get named legend entries

tab_colors = plt.cm.tab10(np.linspace(0, 1, 10))
categorical_groups = {}
for i, cid in enumerate(top_community_ids):
    node_set = louvain_communities[cid]
    categorical_groups[f"Community {cid} (n={len(node_set)})"] = (node_set, tab_colors[i % 10])

other_nodes = set()
for cid, nodes_in_comm in enumerate(louvain_communities):
    if cid not in top_community_ids:
        other_nodes |= nodes_in_comm
if other_nodes:
    categorical_groups["Other communities"] = (other_nodes, "#CCCCCC")

plot_structural_map(
    G_multi, categorical_groups=categorical_groups,
    title="Coimbra -- Community Structure (Louvain)",
    boundary=city_boundary, basemap=False,
    legend_loc="lower left",
    save_path=f"{OUTPUT_DIR}/coimbra_communities.png"
)
print("""
Suggested LaTeX caption:
"Community structure detected via the Louvain method. The eight largest
communities are individually coloured and labelled with their size (number
of nodes); smaller communities are grouped as 'Other communities' for
legibility. Source: authors' own elaboration, generated using Python
(NetworkX/OSMnx) from OpenStreetMap data (\\copyright{} OpenStreetMap contributors)."
""")

## 5. Degree Distribution (larger figure, readable fonts)

In [ ]:
degrees = [d for _, d in G.degree()]

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.hist(degrees, bins=range(1, max(degrees) + 2), edgecolor='black',
        align='left', color='#4C72B0')
ax.set_title("Coimbra -- Degree Distribution", fontsize=16, fontweight='bold')
ax.set_xlabel("Degree", fontsize=13)
ax.set_ylabel("Frequency", fontsize=13)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/coimbra_degree_distribution.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

## 6. Critical Nodes -- likely "Figure 13" flagged as unreadable (REV 04-011)

Previously: tiny default-size scatter points, a legend built from
`ax.get_legend_handles_labels()` deduplication (fragile), no boundary,
no scale bar, default (small) font sizes. Rebuilt using the shared toolkit.

In [ ]:
N = G.number_of_nodes()
top_k = max(5, int(0.01 * N))

top_bet = set(sorted(betweenness_dict, key=betweenness_dict.get, reverse=True)[:top_k])
top_pr = set(sorted(pagerank_dict, key=pagerank_dict.get, reverse=True)[:top_k])
bridge_nodes = set(u for u, v in bridges) | set(v for u, v in bridges)

# Make categories mutually exclusive for a clean legend (previously a node
# in multiple categories silently only showed its last-drawn colour)
only_bet = top_bet - top_pr - bridge_nodes
only_pr = top_pr - top_bet - bridge_nodes
only_bridge = bridge_nodes - top_bet - top_pr
overlap = (top_bet & top_pr) | (top_bet & bridge_nodes) | (top_pr & bridge_nodes)

categorical_groups = {
    "High betweenness only": (only_bet, "crimson"),
    "High PageRank only": (only_pr, "purple"),
    "Bridge node only": (only_bridge, "darkorange"),
    "Multiple criteria (overlap)": (overlap, "black"),
}

plot_structural_map(
    G_multi, categorical_groups=categorical_groups,
    title="Coimbra -- Critical Nodes (Network Vulnerability Points)",
    boundary=city_boundary, basemap=True,
    legend_loc="lower left", figsize=(12, 12),
    save_path=f"{OUTPUT_DIR}/coimbra_critical_nodes.png"
)
print(f"Critical nodes: {len(only_bet)} betweenness-only, {len(only_pr)} PageRank-only, "
      f"{len(only_bridge)} bridge-only, {len(overlap)} overlapping multiple criteria")
print("""
Suggested LaTeX caption:
"Critical nodes identified via betweenness centrality, PageRank, and bridge
participation, shown with mutually exclusive categories and an explicit
legend. Nodes satisfying more than one criterion are shown separately
(black) rather than only displaying the last-applied category. Source:
authors' own elaboration, generated using Python (NetworkX/OSMnx) from
OpenStreetMap data (\\copyright{} OpenStreetMap contributors)."
""")

## 7. Bridges and Weak Ties (with a real legend; "weak tie" labelled explicitly)

Addresses REV 05-04's concern that "weak ties" appears without definition:
the legend entry itself now states the operational definition
(low edge-betweenness), consistent with the Methodology text.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ox.plot_graph(G_multi, ax=ax, node_size=0, edge_color="#CCCCCC",
              edge_linewidth=0.5, show=False, close=False, bgcolor="white")

bridge_line = None
weak_line = None
for u, v in bridges:
    if G_multi.has_edge(u, v):
        x = [G_multi.nodes[u]['x'], G_multi.nodes[v]['x']]
        y = [G_multi.nodes[u]['y'], G_multi.nodes[v]['y']]
        bridge_line, = ax.plot(x, y, color='red', linewidth=2.2, alpha=0.85,
                                label='Topological bridge')

for u, v in weak_ties:
    if G_multi.has_edge(u, v):
        x = [G_multi.nodes[u]['x'], G_multi.nodes[v]['x']]
        y = [G_multi.nodes[u]['y'], G_multi.nodes[v]['y']]
        weak_line, = ax.plot(x, y, color='darkorange', linewidth=1.6, alpha=0.75,
                              label='Weak tie (lowest edge betweenness)')

if city_boundary is not None:
    city_boundary.boundary.plot(ax=ax, color='black', linewidth=1.4, linestyle='--', zorder=5)

handles = [h for h in [bridge_line, weak_line] if h is not None]
ax.legend(handles=handles, loc='lower left', fontsize=12, framealpha=0.9,
          title="Legend", title_fontsize=13)

try:
    ax.add_artist(ScaleBar(85000, units="m", dimension="si-length",
                           location='lower right', box_alpha=0.75, font_properties={'size': 11}))
except Exception as e:
    print(f"Scale bar failed: {e}")

ax.annotate('N', xy=(0.95, 0.12), xytext=(0.95, 0.04), xycoords='axes fraction',
            fontsize=15, fontweight='bold', ha='center',
            arrowprops=dict(facecolor='black', width=3, headwidth=9))

ax.set_title("Coimbra -- Topological Bridges and Weak Ties", fontsize=16, fontweight='bold', pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/coimbra_bridges_weak_ties.png", dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

print("""
Suggested LaTeX caption:
"Topological bridges (red; edges whose removal disconnects the graph) and
weak ties (orange; the 50 edges with lowest edge-betweenness centrality,
Section~X.X) in Coimbra's road network. Source: authors' own elaboration,
generated using Python (NetworkX/OSMnx) from OpenStreetMap data
(\\copyright{} OpenStreetMap contributors)."
""")

## Summary: all figures saved

In [ ]:
print(f"All figures saved to ./{OUTPUT_DIR}/ at {SAVE_DPI} DPI:\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")
print("\nEvery figure above used the same shared toolkit (plot_structural_map),"
      " so labels, legends, colour scales, line widths, boundaries, scale bars,"
      " and titles are consistent across the full figure set, and each has a"
      " ready-to-paste LaTeX caption including a source line.")